# Mastering Google Sheets with the `Sheets` Python Module

Welcome to this comprehensive tutorial on using the `Sheets` Python module! This module provides a simplified, high-level interface for interacting with Google Spreadsheets (Sheets), making it easier to read data into pandas DataFrames.

This tutorial will guide you through setting up authentication, initializing the `Sheets` wrapper, and demonstrating how to effectively read data from your Google Sheets.

## 1. Setup: Install Libraries

First, we need to install the necessary Python libraries. The `Sheets` module relies on `gspread` for Google Sheets API interaction and `pandas` for data manipulation.

In [ ]:
!pip install gspread pandas google-auth --quiet

## 2. The `Sheets` Module Source Code

For this tutorial, we'll embed the `Sheets` module directly into our notebook. Note that the original `Sheets` module depends on `conidk.core.base.Auth` and `conidk.core.base.Config`. Since `conidk` is an external dependency not provided here, we'll create minimal mock classes for `Auth` and `Config` that mimic the expected interface for `gspread` authentication.

In [ ]:
import gspread
from gspread import exceptions
import pandas as pd
from typing import Optional

# --- Mock for conidk.core.base --- 
# These classes are minimal implementations to make the Sheets class runnable
# in this notebook, simulating the expected behavior of conidk.core.base.Auth 
# for gspread authentication.
class Auth:
    """A mock authentication class that wraps gspread's service_account."""
    def __init__(self, service_account_key_path: Optional[str] = None):
        if service_account_key_path:
            self.creds = gspread.service_account(filename=service_account_key_path)
        else:
            raise ValueError("service_account_key_path must be provided for Auth initialization.")

class Config:
    """A placeholder class for configuration, as its attributes are not directly used by Sheets' core logic."""
    pass

# --- Original Sheets Class --- 
class Sheets:
    """A wrapper for Google Sheets API to simplify interactions with spreadsheets."""

    def __init__(
        self,
        sheet_id: str,
        auth: Optional[Auth] = None, # Use our mocked Auth
        config: Optional[Config] = None, # Use our mocked Config
    )->None:
        """Initializes the Sheets wrapper.

        Args:
            sheet_id: The ID or name of the Google Sheet.
            auth: An authentication object.
            config: A configuration object.
        """
        # If auth is not provided, our mock Auth will raise an error
        # which guides the user to provide a service account path.
        self.auth = auth # Auth must be provided and initialized with service account key
        self.config = config or Config()
        self.sheet_id = sheet_id
        
        if not self.auth or not hasattr(self.auth, 'creds'):
            raise ValueError("An authenticated 'Auth' object with 'creds' attribute is required.")

        self.client = gspread.authorize(self.auth.creds)

    def to_dataframe (
        self,
        sheet_name: str = "0"
    ) -> pd.DataFrame:
        """Reads a sheet from the spreadsheet and returns it as a pandas DataFrame.

        Args:
            sheet_name: The name or index of the sheet to read.
        
        Returns:
            A pandas DataFrame containing the sheet data.
            
        Raises:
            FileNotFoundError: If the spreadsheet is not found.
            TypeError: If sheet_name is not a string or an integer.
        """
        try:
            spreadsheet = self.client.open_by_key(self.sheet_id)
        except exceptions.SpreadsheetNotFound:
            spreadsheet = self.client.open(self.sheet_id)

        # Select the worksheet
        if isinstance(sheet_name, str):
            worksheet = spreadsheet.worksheet(sheet_name)
        elif isinstance(sheet_name, int):
            worksheet = spreadsheet.get_worksheet(sheet_name)
        else:
            raise TypeError("sheet_name must be a string (name) or an integer (index).")

        # Get all data as a list of lists
        data = worksheet.get_all_values()

        # Convert to DataFrame
        if not data:
            return pd.DataFrame() # Return empty DataFrame if no data

        # Use the first row as columns, and the rest as data
        return pd.DataFrame(data[1:], columns=data[0])

## 3. Google Sheets API Authentication (Service Account)

To allow your Colab notebook to access your Google Sheets, you need to set up a service account. This is the recommended method for server-to-server interactions without user intervention.

**Steps:**
1.  **Enable Google Sheets API**: Go to the [Google Cloud Console](https://console.cloud.google.com/) and create a new project (or select an existing one). Search for and enable the "Google Sheets API" for your project.
2.  **Create a Service Account**: In the Google Cloud Console, navigate to `IAM & Admin > Service Accounts`. Click `+ CREATE SERVICE ACCOUNT`.
    *   Give it a name (e.g., `sheets-access`).
    *   Grant it the `Viewer` or `Editor` role for Google Drive/Sheets (e.g., `Google Sheets Editor` or `Editor` on the project level for simplicity, but more granular `Editor` on specific files is better for production).
3.  **Create and Download JSON Key**: After creating the service account, click on its email address. Go to the `Keys` tab, click `ADD KEY > Create new key`, select `JSON`, and click `CREATE`. This will download a `.json` file to your computer.
4.  **Share Your Spreadsheet**: The downloaded JSON file contains your service account's credentials. The key step is to share your Google Sheet with the service account's email address (you can find this email on the Service Accounts page in the Google Cloud Console, it typically looks like `your-service-account-name@your-project-id.iam.gserviceaccount.com`). Grant it `Editor` access.

### Upload Your Service Account Key

Now, upload the downloaded `.json` key file to your Colab environment. The `google.colab.files` utility makes this easy.

In [ ]:
from google.colab import files

print("Please upload your service account JSON key file now...")
uploaded = files.upload()

# Assumes only one file is uploaded. Adjust if you upload multiple.
SERVICE_ACCOUNT_KEY_PATH = list(uploaded.keys())[0] 
print(f"\nService account key uploaded: {SERVICE_ACCOUNT_KEY_PATH}")

## 4. Prepare Your Sample Google Sheet

For demonstration purposes, let's assume you have a Google Sheet set up. If not, please create one now:

1.  Go to [Google Sheets](https://docs.google.com/spreadsheets/create) and create a new spreadsheet.
2.  Name it something like "`My Demo Sheet for Sheets Module`".
3.  In the first sheet (e.g., `Sheet1`), add some sample data with a header row. For example:

    | Name    | City       | Age |
    |---------|------------|-----|
    | Alice   | New York   | 30  |
    | Bob     | London     | 24  |
    | Charlie | Paris      | 35  |

4.  (Optional) Add a second sheet and name it `MyData` with different data.
5.  **Get the Spreadsheet ID**: The ID is a long string of characters in the spreadsheet's URL, between `/d/` and `/edit`. 
    Example URL: `https://docs.google.com/spreadsheets/d/YOUR_SPREADSHEET_ID_HERE/edit#gid=0`
6.  **Share the Spreadsheet**: Remember to share this spreadsheet with your service account's email address (from step 3) with `Editor` permissions.

In [ ]:
# Replace with your actual Google Sheet ID and sheet names
YOUR_SPREADSHEET_ID = "YOUR_SPREADSHEET_ID_HERE" # e.g., "1B_T4..........................."
YOUR_SHEET_NAME_1 = "Sheet1" # The default name for the first sheet
YOUR_SHEET_NAME_2 = "MyData" # If you created another sheet with a custom name

## 5. Initialize the `Sheets` Wrapper

Now, let's create an instance of our `Sheets` wrapper. We'll pass the `spreadsheet_id` and our authenticated `Auth` object.

In [ ]:
# Initialize the Auth object with your service account key
sheets_auth = Auth(service_account_key_path=SERVICE_ACCOUNT_KEY_PATH)

# Initialize the Sheets wrapper with your spreadsheet ID and the authenticated Auth object
sheets_wrapper = Sheets(sheet_id=YOUR_SPREADSHEET_ID, auth=sheets_auth)

print(f"Sheets wrapper successfully initialized for spreadsheet ID: {YOUR_SPREADSHEET_ID}")

## 6. Reading Data into a Pandas DataFrame (`to_dataframe`)

The core functionality of the `Sheets` module is reading data from a specified worksheet and converting it into a pandas DataFrame. The `to_dataframe` method allows you to specify the worksheet by its name or its numerical index.

### Reading by Sheet Name

You can easily fetch data by providing the exact name of the worksheet.

In [ ]:
# Read data from the sheet named 'Sheet1' (or YOUR_SHEET_NAME_1)
try:
    df_by_name = sheets_wrapper.to_dataframe(sheet_name=YOUR_SHEET_NAME_1)
    print(f"Data from sheet '{YOUR_SHEET_NAME_1}':")
    print(df_by_name.head())
    print("\nDataFrame Info:")
    df_by_name.info()
except Exception as e:
    print(f"Error reading sheet by name '{YOUR_SHEET_NAME_1}': {e}")

### Reading by Sheet Index

Alternatively, you can access sheets by their zero-based index. The first sheet is at index `0`, the second at `1`, and so on.

In [ ]:
# Read data from the first sheet by its index (0)
try:
    df_by_index = sheets_wrapper.to_dataframe(sheet_name=0)
    print(f"\nData from sheet at index 0 (usually '{YOUR_SHEET_NAME_1}'):")
    print(df_by_index.head())
    print("\nDataFrame Info:")
    df_by_index.info()
except Exception as e:
    print(f"Error reading sheet by index 0: {e}")

### Handling Empty Sheets or Invalid Inputs

The `to_dataframe` method gracefully handles sheets with no data (returning an empty DataFrame). It also raises a `TypeError` for invalid `sheet_name` types.

In [ ]:
# Example of reading an empty sheet (Optional: Create a new sheet named 'EmptySheet' in your spreadsheet and leave it blank)
# Make sure to share 'EmptySheet' with your service account if it's a new sheet.
# try:
#     empty_df = sheets_wrapper.to_dataframe(sheet_name="EmptySheet")
#     print("\nDataFrame from 'EmptySheet':")
#     print(empty_df)
#     print(f"Is DataFrame empty? {empty_df.empty}")
# except Exception as e:
#     print(f"Could not read 'EmptySheet': {e}")

# Demonstrate TypeError for invalid sheet_name type
try:
    # Attempt to read with an unsupported type for sheet_name (e.g., float)
    sheets_wrapper.to_dataframe(sheet_name=1.5) 
except TypeError as e:
    print(f"\nCaught expected error for invalid sheet_name type: {e}")
except Exception as e:
    print(f"Caught an unexpected error: {e}")

## 7. Conclusion

You've successfully explored the `Sheets` Python module, learning how to:

*   Set up Google Sheets API authentication using a service account.
*   Initialize the `Sheets` wrapper with your spreadsheet ID.
*   Read data from Google Sheets into pandas DataFrames by both sheet name and index.
*   Understand how the module handles different scenarios, including invalid inputs.

This module simplifies interactions with Google Sheets, making it a powerful tool for data analysis and automation workflows. Happy coding!